# ORBIT Multi-Turn Non-Tool Session Demo

This notebook demonstrates ORBIT's first linear multi-turn session path without tool calling, using a dual-track benchmark across OpenAI Codex and SSH vLLM.


In [1]:
from pathlib import Path

from orbit.runtime import OrbitCoordinator
from orbit.store import create_default_store
from orbit.runtime.providers.openai_codex import OpenAICodexConfig, OpenAICodexExecutionBackend
from orbit.runtime.providers.ssh_vllm import SshVllmConfig, SshVllmExecutionBackend
from orbit.notebook import NotebookWorkbench

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().resolve().name == 'providers' else Path.cwd().resolve()
TEST_TURNS = [
    'My name is ORBIT tester. Please remember that for this conversation.',
    'What name did I just give you?',
    'Answer in one short sentence.',
]


## Build dual-track backends


In [2]:
codex_backend = OpenAICodexExecutionBackend(
    config=OpenAICodexConfig(model='gpt-5.4'),
    repo_root=REPO_ROOT,
)

vllm_backend = SshVllmExecutionBackend(
    config=SshVllmConfig(
        remote_base_url='http://127.0.0.1:8000/v1',
        model='Qwen/Qwen2.5-Coder-14B-Instruct-AWQ',
        api_key='EMPTY',
        auto_tunnel=True,
        ssh_host='ws3',
        local_port=8000,
        remote_host='127.0.0.1',
        remote_port=8000,
    )
)


## Helper to run one multi-turn session track


In [3]:
def run_track(backend, backend_name: str, model_name: str):
    coordinator = OrbitCoordinator(
        store=create_default_store(),
        workspace_root=REPO_ROOT,
        backend=backend,
    )
    workbench = NotebookWorkbench(coordinator)
    session = coordinator.create_session(backend_name=backend_name, model=model_name)
    plans = []
    for turn in TEST_TURNS:
        plans.append(coordinator.run_session_turn(session_id=session.session_id, user_input=turn))
    return {
        'coordinator': coordinator,
        'workbench': workbench,
        'session': session,
        'plans': plans,
    }


## Run Codex track


In [7]:
codex_result = run_track(codex_backend, 'openai-codex', 'gpt-5.4')
[plan.plan_label for plan in codex_result['plans']]


['openai-codex-final-text',
 'openai-codex-final-text',
 'openai-codex-final-text']

## Run vLLM track


In [4]:
vllm_result = run_track(vllm_backend, 'ssh-vllm', 'Qwen/Qwen2.5-Coder-14B-Instruct-AWQ')
[plan.plan_label for plan in vllm_result['plans']]


['ssh-vllm-final-text', 'ssh-vllm-final-text', 'ssh-vllm-final-text']

## Compare transcript tables


In [8]:
codex_result['workbench'].session_messages_dataframe(codex_result['session'].session_id)


,message_id,session_id,turn_index,role,content,created_at,provider_message_id,metadata
0,msg_55b6390081dc,session_c340bd51f8c2,1,user,My name is ORBIT tester. Please remember that ...,2026-04-01 21:44:55.928397+00:00,None,{}
1,msg_ded3217d8e55,session_c340bd51f8c2,2,assistant,Got it — I’ll remember that your name is ORBIT...,2026-04-01 21:44:57.296065+00:00,None,"{'source_backend': 'openai-codex', 'plan_label..."
2,msg_a0a198b1a5e1,session_c340bd51f8c2,3,user,What name did I just give you?,2026-04-01 21:44:57.299945+00:00,None,{}
3,msg_d8b85d2733db,session_c340bd51f8c2,4,assistant,You said your name is ORBIT tester.,2026-04-01 21:44:58.649387+00:00,None,"{'source_backend': 'openai-codex', 'plan_label..."
4,msg_6a269d9f0a10,session_c340bd51f8c2,5,user,Answer in one short sentence.,2026-04-01 21:44:58.654471+00:00,None,{}
5,msg_61799e4325a4,session_c340bd51f8c2,6,assistant,Your name is ORBIT tester.,2026-04-01 21:44:59.634682+00:00,None,"{'source_backend': 'openai-codex', 'plan_label..."


In [6]:
vllm_result['workbench'].session_messages_dataframe(vllm_result['session'].session_id)


,message_id,session_id,turn_index,role,content,created_at,provider_message_id,metadata
0,msg_bae4060805cb,session_8270d47f42f7,1,user,My name is ORBIT tester. Please remember that ...,2026-04-01 21:44:46.217739+00:00,None,{}
1,msg_62bf74a32fed,session_8270d47f42f7,2,assistant,Hello ORBIT tester! How can I assist you today?,2026-04-01 21:44:47.064118+00:00,None,"{'source_backend': 'ssh-vllm', 'plan_label': '..."
2,msg_fe367ea8e1f0,session_8270d47f42f7,3,user,What name did I just give you?,2026-04-01 21:44:47.071928+00:00,None,{}
3,msg_a58b4de420f5,session_8270d47f42f7,4,assistant,"You just gave me the name ""ORBIT tester.""",2026-04-01 21:44:47.780114+00:00,None,"{'source_backend': 'ssh-vllm', 'plan_label': '..."
4,msg_df39b4d9c3ab,session_8270d47f42f7,5,user,Answer in one short sentence.,2026-04-01 21:44:47.784616+00:00,None,{}
5,msg_9038e275e0db,session_8270d47f42f7,6,assistant,"You named me ""ORBIT tester.""",2026-04-01 21:44:48.425617+00:00,None,"{'source_backend': 'ssh-vllm', 'plan_label': '..."


In [ ]:
codex_result['workbench'].session_turn_summary_dataframe(codex_result['session'].session_id)


In [ ]:
vllm_result['workbench'].session_turn_summary_dataframe(vllm_result['session'].session_id)
